# Lab 01 · Better prompts and small evaluations

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## Give the model the policy, then test it

**So far:** in Lab 00 the model answered without seeing the policy, and it got the mixed cases wrong.

**New:** a *guided prompt* that contains the policy, a *parser* that rejects badly formatted replies, and a small *evaluation* that scores the model on five labelled cases.

A guided prompt has four labelled parts: the policy, one worked example, the case, and the exact reply format. JSON lets our code read the decision reliably. The one-sentence reason is for a person to check. Asking the model to check both criteria is a short, visible form of step-by-step reasoning. We do not ask for a long reasoning trace.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Build a guided prompt

`guided_prompt(case)` fills the same four-part template for any case. The worked example deliberately uses a case where only **one** criterion holds (20 people, but no independent deploys), because that is exactly where Lab 00's model went wrong. `json.dumps(case)` writes the case as JSON, so the model sees `false` in the same style as the worked example.

In [ ]:
import json

POLICY = ("Choose microservices only if team_size >= 8 AND independent_deploys is true; "
          "otherwise choose modular monolith.")
WORKED_EXAMPLE = ('Case: {"team_size": 20, "independent_deploys": false}\n'
                  'Reply: {"decision": "modular monolith", "reason": "20 people, but no independent deploys."}')

def guided_prompt(case):
    return (f"Policy: {POLICY}\n\n"
            f"Worked example:\n{WORKED_EXAMPLE}\n\n"
            f"Case: {json.dumps(case)}\n"
            'Check both criteria, then reply with JSON only: {"decision": "...", "reason": "one sentence"}')

print(guided_prompt({"team_size": 3, "independent_deploys": False}))

## Step 2 · Parse the reply, and reject what breaks the format

A model can ignore the format. `parse_reply` returns the decision when the reply is valid JSON with a known label. Otherwise it raises a `ValueError` that says what is wrong. The `try`/`except` loop below tries it on one good reply and two bad ones. Rejecting bad output loudly is safer than guessing what the model meant.

In [ ]:
LABELS = ("modular monolith", "microservices")

def parse_reply(reply):
    """Return the decision from a JSON reply. Raise ValueError if the reply breaks the format."""
    try:
        data = json.loads(reply)
    except json.JSONDecodeError:
        raise ValueError("not JSON") from None
    decision = str(data.get("decision", "")).strip().lower()
    if decision not in LABELS:
        raise ValueError(f"unknown decision {decision!r}")
    return decision

for reply in ['{"decision": "Modular Monolith", "reason": "3 people."}', "Microservices!", '{"decision": "serverless"}']:
    try:
        print("accepted:", parse_reply(reply))
    except ValueError as error:
        print("rejected:", error)

## Step 3 · Write a labelled test set

Each test has a case, the label the policy expects, and a note on why the case is in the set. Together the five cases cover both criteria, each criterion on its own, and the boundary value of exactly 8 people.

In practice mode we also need a scripted reply for each test. One scripted label is deliberately wrong, so you can see what a failing row looks like.

In [ ]:
tests = [  # (case, expected label, why this case is in the set)
    ({"team_size": 3,  "independent_deploys": False}, "modular monolith", "neither criterion"),
    ({"team_size": 12, "independent_deploys": True},  "microservices",    "both criteria"),
    ({"team_size": 9,  "independent_deploys": False}, "modular monolith", "big team only"),
    ({"team_size": 5,  "independent_deploys": True},  "modular monolith", "independent deploys only"),
    ({"team_size": 8,  "independent_deploys": True},  "microservices",    "boundary: exactly 8"),
]
# Scripted practice labels, one per test. The third one is a deliberate mistake.
scripted = ["modular monolith", "microservices", "microservices", "modular monolith", "microservices"]
print(len(tests), "tests,", len(scripted), "scripted replies")

## Step 4 · Score the prompt

`zip(tests, scripted)` pairs each test with its scripted reply. Every reply ends up in one of three buckets: **pass** (right label), **fail** (wrong label), or **invalid** (the parser rejected it). The final `assert` is a *quality gate*: the run stops with an error if fewer than 80% of the tests pass.

In [ ]:
counts = {"pass": 0, "fail": 0, "invalid": 0}
for (case, expected, why), label in zip(tests, scripted):
    example = json.dumps({"decision": label, "reason": "Scripted practice reply."})
    try:
        decision = parse_reply(ask(guided_prompt(case), example=example))
        verdict = "pass" if decision == expected else "fail"
    except ValueError as error:
        decision, verdict = f"({error})", "invalid"
    counts[verdict] += 1
    print(f"{verdict.upper():<8} {why:<25} got {decision!r}, expected {expected!r}")
accuracy = counts["pass"] / len(tests)
print(counts, f"accuracy: {accuracy:.0%}")
assert accuracy >= 0.8, "Below the 80% quality gate. Read the FAIL and INVALID rows."

## What you should see

Step 2 prints `accepted: modular monolith`, then two `rejected:` lines. Step 4 prints five rows. Four are `PASS`, and the `big team only` row is `FAIL`, because its scripted reply says `microservices`. The last line is `{'pass': 4, 'fail': 1, 'invalid': 0} accuracy: 80%`, which just meets the gate.

Practice scores only test the evaluation code, not a model. In live mode, the score shows how this prompt did on these five cases. Five cases cannot prove that a prompt is good in general.

## Try it

1. Fix the deliberate mistake: change the third scripted label to `"modular monolith"`. Accuracy becomes 100%.
2. Change the **first** scripted label to `"serverless"`. Its row becomes `INVALID`, and the gate stops the run, because only 3 of 5 tests pass.
3. Add a test for 7 people with independent deploys, and a scripted label for it. Which bucket does it land in?
4. In live mode, compare this score with Lab 00's agreement count. Which part of the guided prompt made the difference?

## Takeaway

A clear prompt, a strict parser, and a labelled test set turn "the answer looks right" into a number you can track. Every later lab keeps this habit: it checks the model's output before trusting it. Next, Lab 02 stops pasting the policy into the prompt and retrieves it from a store instead.

## Read further

- [Chain-of-thought prompting](https://arxiv.org/abs/2201.11903)
- [Prompt engineering](https://developers.openai.com/api/docs/guides/prompt-engineering)